# Lesson 4 · Three knobs to tame every sound — the frequency, amplitude, and phase (φ) of a sine wave

**Learning goals**
1. Master what each of the three parameters in `A · sin(2πft + φ)` controls
2. Implement `sinusoid(t, A, f, phi)` and verify it with assert
3. Understand frequency superposition: a chord = several sine waves added together
4. Lay the groundwork for L05 (complex numbers) and L06 (Euler's formula)

**Aurora connection (one line to remember for now)**: the sine generator in the repo is **the same formula** as this lesson's, but with a different interface (details in the 🔗 cell after the exercise).
L37's DFT will break any signal into a sum of sine waves — and the "atoms" it breaks into are exactly this lesson's `sinusoid`.

**Carried over from L03**: every bright horizontal line on a spectrogram corresponds to a sine wave at one particular frequency.
In this lesson you take control of that line's position (frequency), brightness (amplitude), and starting angle (phase) yourself.

← **Previous lesson** · [L03 · Spectrogram intuition](../0_foundation/L03_spectrogram.ipynb)

> Last lesson covered **spectrogram intuition**: learning to read the three axes of a time–frequency plot before learning the FFT.
> This lesson is about **the three ingredients of a sine wave**.

## This lesson's story: three knobs control everything

Picture an old radio tuner. It has three knobs:

- **Volume knob** (amplitude A): turn it up → louder; turn it down → softer. Ranges from 0 to infinity.
- **Frequency knob** (frequency f): turn it up → higher pitch (like a soprano); turn it down → lower (like a subwoofer).
- **Time-shift knob** (phase φ): slides the whole sound a little earlier or later, so it no longer starts at a peak but somewhere in between.

A sine wave is completely determined by these three knobs:

| Knob | Symbol | Physical meaning | Effect on the spectrogram |
|---|---|---|---|
| Volume | A (amplitude) | How loud → height of the peaks | Brighter color |
| Frequency | f (Hz) | How fast → cycles per unit time | Line sits higher |
| Phase | φ (rad) | Where it starts → where the wave begins | Spectrogram unchanged; the waveform shifts sideways |

Formula: `x(t) = A · sin(2π·f·t + φ)`

In this lesson you'll implement `sinusoid(t, A, f, phi)` — one of the basic building blocks of the whole Aurora DSP pipeline.

## 1. sin and cos: two sides of the same circle

Ever wondered why sin and cos look so alike, yet aren't quite the same?

Put a point on the unit circle and let it turn counterclockwise at a steady speed. At any moment:
- **cos(θ)** = its "shadow" on the x axis (horizontal)
- **sin(θ)** = its "shadow" on the y axis (vertical)

cos leads sin by exactly π/2 (90°). In other words:

```
cos(θ) = sin(θ + π/2)
```

Two functions, one rotation, two different directions of projection. Later, complex numbers (L05) will pack these two shadows into a single number — and then you'll find that Euler's formula `e^{iθ} = cos θ + i·sin θ` describes exactly this rotating point.

> 📌 If two questions popped up as you read this — "Where's that circle? I want to see a picture" and "Does 'leads' mean adding π/2, or the curve moving left — which way is it?" — hang on. The second code cell below actually draws the unit circle and its shadows, and lines up the three ways of saying "leads" in one go. Keep going for now.

## Into the lab: angles, coordinates, and rotation

sin and cos describe the two components of the same circular motion: cos is a quarter cycle ahead of sin. Look at where the peaks, zeros, and troughs of the two curves sit relative to each other, and confirm that cos(0)=1 while sin(0)=0.

In [ ]:
# Aurora matplotlib bootstrap
from pathlib import Path
import sys

_root = None
_cwd = Path.cwd().resolve()
for _candidate in (_cwd, *_cwd.parents):
    if (_candidate / '_matplotlib_bootstrap.py').exists():
        _root = _candidate
        break
if _root is None:
    _notebooks_dir = _cwd / 'notebooks'
    if _notebooks_dir.exists():
        for _found in _notebooks_dir.rglob('_matplotlib_bootstrap.py'):
            _root = _found.parent
            break
if _root is not None and str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

from _matplotlib_bootstrap import apply as _aurora_mpl_apply
_aurora_mpl_apply()


In [ ]:
import numpy as np, matplotlib.pyplot as plt
t = np.linspace(0, 2*np.pi, 400)
plt.plot(t, np.sin(t), label='sin')
plt.plot(t, np.cos(t), label='cos')
plt.legend(); plt.title('sin & cos'); plt.grid(True, alpha=.3); plt.show()

### One more picture: where exactly is that "circle"? And which way does "leads" go?

The plot above has only two curves; the "point turning on the unit circle" from Section 1 hasn't shown up yet. The next code cell draws it: on the left, the circle and the point, with two dashed lines showing the point's "shadows" on the x and y axes; on the right, the familiar sin/cos curves. The two plots say **the same thing** — the curves on the right are a record of the two shadows on the left, unrolled as the angle θ grows.

Now let's settle something that's easy to get tangled up in: what does "cos leads sin by exactly π/2" really mean? The formula `cos(θ) = sin(θ + π/2)` has a **plus sign**, yet in the picture cos's peak is to the **left** of sin's peak — so which way counts as "leading"?

These are actually three ways of saying the same thing. Let's line them up:

1. **The event view**: think of "reaching the peak" as an event. cos reaches its peak at θ=0; sin has to wait until θ=π/2. Same event, and cos gets there **earlier** — that's what "leading" means.
2. **The picture view**: happening earlier = every event along the curve happens sooner = the whole curve **shifts left** by π/2. So cos's peak sitting to the left of sin's peak is no contradiction at all.
3. **The formula view**: shifting the curve left by π/2, translated into the argument, means "the value of cos at θ = the value sin reaches ahead of time at θ+π/2", i.e. `cos(θ) = sin(θ + π/2)`. The plus sign goes on **the argument inside the parentheses**, and its effect is to move the curve **to the left** — this is the point beginners most often get backwards. Check it by plugging in θ=0: `sin(0 + π/2) = 1 = cos(0)`. It matches.

One-line summary: **leading = the event happens earlier = the curve shifts left = add π/2 to the argument**. All four phrasings point to the same π/2.

In [ ]:
# Actually draw the "shadows": left, the unit circle and the turning point; right, the two projections unrolled against the angle
import numpy as np
import matplotlib.pyplot as plt

theta = np.pi / 3            # an example: the point has turned to 60° (try another angle and rerun)
circle = np.linspace(0, 2*np.pi, 200)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.5))

# ---- Left: unit circle + turning point + two dashed "shadow" lines ----
ax1.plot(np.cos(circle), np.sin(circle), color='gray', lw=1)   # the circle itself
x0, y0 = np.cos(theta), np.sin(theta)
ax1.plot([0, x0], [0, y0], 'k-', lw=1.5)                       # radius from the center to the point
ax1.plot(x0, y0, 'ro', ms=8)                                   # the turning point
ax1.plot([x0, x0], [0, y0], 'b--', lw=1)                       # vertical dashed line: the point dropped straight down to the x axis
ax1.plot([0, x0], [y0, y0], 'g--', lw=1)                       # horizontal dashed line: the point moved across to the y axis
ax1.plot(x0, 0, 'bs', ms=7)
ax1.annotate('cos θ (shadow on the x axis)', (x0, 0), textcoords='offset points',
             xytext=(5, -15), color='b')
ax1.plot(0, y0, 'g^', ms=7)
ax1.annotate('sin θ (shadow on the y axis)', (0, y0), textcoords='offset points',
             xytext=(-10, 8), color='g', ha='right')
ax1.axhline(0, color='gray', lw=.5); ax1.axvline(0, color='gray', lw=.5)
ax1.set_aspect('equal'); ax1.set_title('A point on the unit circle (θ = π/3) and its two shadows')

# ---- Right: sin/cos curves, with an arrow marking "same peak, cos gets there π/2 earlier" ----
tt = np.linspace(0, 2*np.pi, 400)
ax2.plot(tt, np.sin(tt), 'g', label='sin')
ax2.plot(tt, np.cos(tt), 'b', label='cos')
ax2.plot(0, 1, 'bo')          # cos's peak: already there at θ=0
ax2.plot(np.pi/2, 1, 'go')    # sin's peak: not until θ=π/2
ax2.annotate('', xy=(0, 1.15), xytext=(np.pi/2, 1.15),
             arrowprops=dict(arrowstyle='->', color='r'))
ax2.text(np.pi/4, 1.22, "cos's peak arrives π/2 earlier", color='r', ha='center')
ax2.set_ylim(-1.4, 1.5); ax2.legend(); ax2.grid(alpha=.3)
ax2.set_title('"Leading" = the same event (the peak) happens earlier')
plt.tight_layout(); plt.show()

## Hands-on: four special angles

At the four angles 0, π/2, π, 3π/2, the values of cos and sin are all 0 or ±1, so you can check them in your head. After running, compare `z.real` and `z.imag` and confirm the output matches your expectations exactly.

> ⚠️ **Let's be clear up front: the next code cell borrows two "props from the next lessons." You haven't missed anything.**
>
> 1. **What is `1j`?** It's how Python writes "the imaginary unit i" (engineers use j so it isn't confused with electric current, and Python kept the habit; it has to have a number in front, so you write `1j`, not `j`). Complex numbers are the star of **L05**; you don't need to understand them today.
> 2. **What is `np.exp(1j * θ)`?** Today, treat it as a **black-box machine**: feed in an angle θ and it hands back that angle's cos and sin at the same time — `z.real` (the real part) is cos(θ), and `z.imag` (the imaginary part) is sin(θ). **Why** e raised to an "imaginary power" gives cos and sin is the mystery **L06, Euler's formula**, will reveal; this lesson doesn't ask you to understand it at all.
>
> So why bring it on stage now? Because it's exactly the code version of "the point turning on the unit circle": one θ goes in, and both shadows of that point on the circle come out. Today you only need to **use its output to check the cos/sin values you worked out in your head in the previous section**; we'll take this machine's "user manual" apart in L05/L06.

In [ ]:
import numpy as np

angles = np.array([0, np.pi/2, np.pi, 3*np.pi/2])
z = np.exp(1j * angles)

print('angles =', np.round(angles, 3))
print('real part cos =', np.round(z.real, 3))
print('imaginary part sin =', np.round(z.imag, 3))
print('complex z =', np.round(z, 3))


## Code experiment: sweep through frequency, amplitude, and phase

With a sample rate (sr) of sr = 16 Hz and a duration of 0.5 s, you get only 8 samples — few enough to check one by one: when freq doubles, the largest change between neighboring samples gets noticeably bigger (about 1.85× for f=1→2, about 1.41× for f=2→4 — a sine isn't a straight line, so it doesn't exactly double); when amplitude doubles, the output's min/max scale up by the same factor.

In [ ]:
import numpy as np

sample_rate = 16
duration = 0.5
t = np.arange(round(duration * sample_rate)) / sample_rate

for freq in [1, 2, 4]:
    wave = np.sin(2 * np.pi * freq * t)
    print(f'freq={freq}Hz ->', np.round(wave, 3))

for amplitude in [0.5, 1.0, 2.0]:
    wave = amplitude * np.sin(2 * np.pi * 2 * t)
    print(f'amplitude={amplitude} -> min={wave.min():.1f}, max={wave.max():.1f}')


### Where do those two numbers (1.85, 1.41) actually come from?

The note above tossed out "about 1.85× for f=1→2, about 1.41× for f=2→4," but the code only printed the waveforms themselves and never computed those ratios — you shouldn't have to invent the steps yourself. The next cell fills in the full calculation, using a new friend:

- **`np.diff(wave)`**: subtracts **each pair of neighboring numbers** in the array, giving "how far each step went." For example, `diff([0, 0.4, 0.7])` → `[0.4, 0.3]`.
- Take the largest absolute value of that, and you have "the largest change between neighboring samples."

When you run it, you'll **see** 1.85 and 1.41 right there in the output, no manual subtraction needed.

**Why doesn't this "biggest step" exactly double when the frequency doubles?** Think of a ceiling: a sine's value always stays between −1 and +1, so the difference between two neighboring samples is **at most 2** (one jump from the trough at −1 to the peak at +1). It can't keep doubling forever. The higher the frequency and the sparser the samples, the closer this "biggest step" gets to the ceiling, so it grows more slowly each time: 1.85 → 1.41 → … getting closer and closer to 1. That's what "a sine is curved, so it doesn't exactly double" really means — **for a straight line, doubling the slope doubles the difference, but a sine is capped at ±1, and near the peaks the change "saturates."**

One more secret revealed: in the output, the f=4 row's sample values are only 0 and ±1 (only 4 samples per cycle, landing exactly on peak, trough, and zeros), so the biggest step is 1.0; for f=2 the biggest step is 0.707 (the familiar √2/2 from sin 45°). So 1.41 ≈ 1.0/0.707 = √2 — if you guessed it looks like √2, you guessed right. It grows straight out of the sine values at these special angles; it's not a coincidence and not a typo.

In [ ]:
# Actually compute "the largest change between neighboring samples" — the 1.85 and 1.41 in the note above come from here
import numpy as np

sample_rate = 16
duration = 0.5
t = np.arange(round(duration * sample_rate)) / sample_rate

prev_max_step = None
for freq in [1, 2, 4]:
    wave = np.sin(2 * np.pi * freq * t)
    steps = np.diff(wave)              # diff = "subtract neighboring numbers": wave[1]-wave[0], wave[2]-wave[1], ...
    max_step = np.max(np.abs(steps))   # between two neighboring samples, how far the biggest step went
    line = f'freq={freq}Hz | neighbor diffs = {np.round(steps, 3)} | biggest step = {max_step:.3f}'
    if prev_max_step is not None:
        line += f' | {max_step/prev_max_step:.2f}× the previous row'
    print(line)
    prev_max_step = max_step

## 2. Nailing down the formula: `x(t) = A · sin(2π·f·t + φ)` (the three knobs were covered in the story cell)

- **A, amplitude** = how loud (stretches the `[-1, 1]` output to `[-A, A]`)
- **f, frequency** = how high, in Hz (complete oscillations per second; sets the pitch)
- **φ, phase** = the starting angle offset, in radians (with φ=π/2 the waveform starts at a peak)

The three parameters are orthogonal: A only scales the size, f only changes how fast it oscillates, φ only shifts the starting point — changing one doesn't affect the other two.
When you synthesize complex waveforms (chords, and later the individual components of a filterbank), you can turn each component's knobs separately.

## 3. ✏️ Implement `sinusoid(t, A, f, phi)`

**Line of reasoning**:
1. `t` is the time array (seconds), and `f` is complete cycles per second (Hz). One complete cycle is `2π` radians, so the angle at time `t` is `2π·f·t`.
2. Add the initial phase `phi` (radians) to get the full sequence of angles `2π·f·t + phi`.
3. `np.sin(...)` maps the angles to `[-1, 1]`; multiplying by `A` stretches the range to `[-A, A]`. Mind the order: compute the angles first, then multiply by A once, to avoid repeated scalar multiplications.

**Reference input and output**: `A=2, f=1, phi=0, t=[0, 0.25, 0.5, 0.75]` → `[0, 2, 0, -2]` (four samples across one cycle)

> 💡 Need a hint? Once you've finished the exercise, you can look at the reference implementation in the `solutions/` folder.


### Before writing code, fill in the variable table

Get three things clear before you write `sinusoid`:
- Inputs: `t` (time array), `A` (amplitude), `f` (frequency in Hz), `phi` (initial phase, default 0)
- Key step: compute the phase sequence `2π·f·t + phi`, then multiply by A
- Returns: a float array with the same shape as `t`, with values in `[-A, A]`

In [ ]:
def sinusoid(t, A, f, phi=0.0):
    # ✏️ TODO: return the sine wave
    raise NotImplementedError("TODO: implement sinusoid(t, A, f, phi)")


In [ ]:
import numpy as np
t = np.linspace(0, 1, 8, endpoint=False)
try:
    y = sinusoid(t, 2.0, 1.0, 0.0)
    assert np.allclose(y, 2*np.sin(2*np.pi*1.0*t)), 'should equal 2·sin(2πt)'
    # Extra check: with phi=π/2 it should equal cos
    y_cos = sinusoid(t, 1.0, 1.0, np.pi/2)
    assert np.allclose(y_cos, np.cos(2*np.pi*t), atol=1e-10), 'with phi=π/2 it should equal cos'
    print('✅ Passed: you can now build a sine wave with any amplitude/frequency/phase.')
except (NotImplementedError, TypeError):
    print('⚠️ sinusoid() is not implemented yet; finish the TODO above before running this cell.')


**🔗 Aurora connection**: `aurora.audio.sine` (see `src/aurora/audio/io.py`) uses the same formula, but with a different interface — it takes `freq, duration, sample_rate` instead of a time array `t`, and has no `phi` parameter. You'll implement a fully matching `my_sine` in L33. (The FFT's output is complex numbers — that's for the next lesson, L05; we won't go into it here.)

### Coming up: the next cell "counts how many times the waveform crosses the zero line" — meet three new friends first

Next we'll switch to a different way of checking frequency: **instead of looking at the waveform's shape, count how many times it crosses the zero line (the horizontal axis)**. The intuition: the higher the frequency, the more often the wave swings up and down, so naturally it crosses the zero line more often.

But how do you write "count the crossings" in code? The cell below uses the one-liner `np.sum(np.diff(np.signbit(x)) != 0)`, which brings in three new faces at once. Let's unpack it with a tiny 4-number example first — say `x = [0.5, -0.2, -0.7, 0.3]` (it crosses the zero line 2 times: once going down, once going up):

1. **`np.signbit(x)`**: "Is this number negative?" — answered one by one, giving `[False, True, True, False]` (False = non-negative, True = negative).
2. **`np.diff(...)`**: an old friend, "subtract neighbors." If two neighboring answers are **the same** (no sign change), the difference is 0; if they're **different** (the sign changed, meaning the wave went from above the zero line to below it, or the other way round), the difference isn't 0.
3. **`np.sum(... != 0)`**: `!= 0` marks the "sign changes" as True; when you sum True/False, True counts as 1 and False as 0 — so the total is **how many times the sign changed**, i.e. how many times it crossed the zero line. This example gives 2, which is right.

In one sentence: **number of sign changes = number of zero-line crossings**. Go into the code below expecting that "in theory, an f Hz wave should cross zero about 2f times per second," and see where the counted number differs from that expectation — the small difference is exactly what the next section explains.

In [ ]:
sr = 32
duration = 1.0
t = np.arange(round(duration * sr)) / sr

for freq in [1, 2, 4, 8]:
    x = np.sin(2*np.pi*freq*t)
    zero_crossings = np.sum(np.diff(np.signbit(x)) != 0)
    print(f'freq={freq:>2}Hz | first 8 samples={np.round(x[:8], 2)} | zero crossings≈{zero_crossings}')


## Parameter experiment: turn one knob at a time

Change `f` from 1 to 4 and watch the number of zero crossings over the same range of `t` go from 1 to 7 — the pattern is about 2f−1 (f=1,2,4,8 gives 1,3,7,15). f went up 4×, but the count went from 1 to 7 — it isn't simple 4× proportional growth; only as f gets bigger does "double the frequency → roughly double the crossings" really hold. Then change `phi` to `np.pi/2` and confirm the waveform starts with a cosine shape (hitting the peak first).

### Why 2f−1 and not the 2f you'd expect? Where did that "−1" go?

Your intuition is right: an f Hz sine has f complete cycles in one second, and each cycle **crosses the zero line once going down and once going up**, so in theory it should cross **2f times**. Yet the code always counts 2f−1 — exactly one short every time. The missing crossing isn't magic; let's check f=1 point by point:

- Within one second, sin(2πt) has zeros at three places: **t = 0, 0.5, 1.0**.
- **The zero at t = 0 doesn't get counted**: it's the first sample, there's no earlier sample to compare it with for "did the sign change," so "starting here" doesn't count as a crossing (the wave **starts** at zero; it doesn't **cross** zero).
- **The crossing at t = 1.0 happens outside the window**: the last sample of `t = np.arange(32)/32` is t = 31/32 ≈ 0.969, so the sampling window is [0, 1), **which doesn't include 1.0**. The wave really does cross zero again at t=1.0, but that belongs to "the next second," and this second's array can't see it.
- All that actually gets counted is the one at t = 0.5 → **1 = 2×1 − 1**. ✓

The general rule: at f Hz the zeros are at t = 0, 1/(2f), 2/(2f), …, 1.0. Drop the first (the starting point isn't a crossing) and the last (1.0 is outside the window), and you're left with **2f−1**. So the "−1" comes from **the way we count and the edges of the sampling window**, not from the wave itself oscillating one time fewer.

### That line "it isn't simple 4× proportional growth," in plain words

The sentence is trying to rule out this misreading: "f goes from 1 to 4 (×4), so the zero-crossing count should be exactly ×4 too, i.e. 1×4=4" — but the actual count is 7. The plain version: **the theoretical crossing count 2f is the one that's proportional to f** (double f and it doubles, ×4 f and it's ×4); the 2f−1 the code counts is just always 1 short, so at small f the ratios look "not proportional" (1→3 is 3×, 1→7 is 7×). The bigger f gets, the less that −1 matters, and the closer the ratio gets to the theoretical proportional relationship. The rule didn't change; it was just thrown off by that "−1" remainder.

### Try changing phi yourself

The code cell above doesn't actually have a `phi` variable (it's fixed at `np.sin(2πft)`), so there's nothing to change even if you wanted to. Below is an extra cell, a version that **really has a phi knob**, and it marks every zero crossing on the plot with a red cross, so "the counted number" becomes something you can see and touch. Try changing `f` to 2 and 4, then `phi` to `np.pi/2`, and verify the explanation above one step at a time.

> 🎁 Easter egg: after you change `phi` to `np.pi/2` (with f=1), the zeros move from 0, 0.5, 1.0 to 0.25, 0.75 — **no longer sitting on the window edges** — and this time you'll count the full 2f = 2! That proves it from the other direction: "the −1 comes from the edges, not from the wave itself."

In [ ]:
# Zero-crossing counter with a phi knob: change f and phi, and red crosses mark every crossing that gets counted
import numpy as np
import matplotlib.pyplot as plt

sr = 32
duration = 1.0
t = np.arange(round(duration * sr)) / sr

f   = 1          # ← try changing to 2, 4, 8
phi = 0.0        # ← then try np.pi/2: the waveform starts at a peak (it becomes a cos shape)

x = np.sin(2 * np.pi * f * t + phi)

is_negative  = np.signbit(x)              # is each sample negative? → a True/False array
sign_changed = np.diff(is_negative) != 0  # where neighboring answers differ = sign change = crossing the zero line
crossings    = np.sum(sign_changed)       # True counts as 1, False as 0; the total is the number of crossings

plt.figure(figsize=(9, 3))
plt.plot(t, x, '.-', lw=.8)
plt.axhline(0, color='gray', lw=.8)
cross_idx = np.where(sign_changed)[0]     # which positions the sign changes happen at
plt.plot(t[cross_idx + 1], x[cross_idx + 1], 'rx', ms=9, label=f'zero crossings counted: {crossings} times')
plt.title(f'f={f} Hz, phi={phi:.2f} | counted {crossings} (theoretical zeros ≈ 2f = {2*f})')
plt.legend(); plt.grid(alpha=.3); plt.show()

print(f'f={f} Hz, phi={phi:.2f} -> zero crossings = {crossings} (theory 2f = {2*f}; for where the difference comes from, see "where did the −1 go" above)')

## Parameter experiment: chord = several sine waves added together

Any complex sound can be broken down into a sum of sine waves (this is the core idea of Fourier analysis).
Going the other way, add several `sinusoid`s of different frequencies together and you can synthesize a chord.

> ⚠️ **Heads-up before you run this: the labels on the plot below are real, but the shapes are an "illusion."**
>
> If you use the formula from Question 3 of this lesson's whiteboard challenge (samples per cycle = sr/f) to check the code below, you'll get 200/261.6 ≈ **0.76 samples per cycle** — not even one sample per cycle! If you've already worked that out and grown suspicious: **congratulations, your check is completely right, and the plot really does "go wrong"** — on purpose, as a lesson for you to see.
>
> When samples are too sparse, something called **aliasing** happens: the real high-frequency wave "slips through" the sparse samples, and the curve connecting the dots looks like **a much lower, fake frequency**. Think of the wagon wheels in old movies: when the wheel spins too fast and the camera doesn't take enough frames per second, the wheel seems to turn slowly or even backwards. For this cell's code specifically (sr=200):
>
> - 261.6 Hz disguises itself as about **61.6 Hz** (261.6 − 200)
> - 329.6 Hz disguises itself as about **70.4 Hz** (400 − 329.6)
> - 392.0 Hz disguises itself as about **8 Hz** (400 − 392) — which is why that "G4" looks like a slow wave swinging only 8 times a second
>
> For the waveform to tell the truth, the rule of thumb is that **the sample rate must be more than 2 times the signal's highest frequency** (this is called the Nyquist condition; **L34 aliasing** uses experiments to explain why; L07 is about the intuition of Fourier superposition, not a dedicated lesson on the sampling theorem). Likewise, the closing line "the DFT breaks the chord back into C4/E4/G4" can't happen at sr=200 either — the DFT can only see the three disguised fake frequencies.
>
> So why keep this sr=200 code? Because it demonstrates the **idea** of "stacking up a chord" (adding a few waves point by point) correctly, and it makes a perfect cautionary example. Run it first and see what the illusion looks like; right after, in the next cell, we'll raise sr to 8000 and see the real C4/E4/G4.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

sr, dur = 200, 1.0
t = np.arange(round(dur * sr)) / sr

try:
    # C major chord: C4 + E4 + G4
    c4 = sinusoid(t, A=1.0, f=261.6)
    e4 = sinusoid(t, A=0.8, f=329.6)
    g4 = sinusoid(t, A=0.6, f=392.0)
    chord = (c4 + e4 + g4) / 3.0  # divide by 3 so the three amplitudes added together don't go outside [-1,1]

    fig, axes = plt.subplots(4, 1, figsize=(9, 7), sharex=True)
    for ax, sig, label in zip(axes,
            [c4, e4, g4, chord],
            ['C4 (261.6 Hz)', 'E4 (329.6 Hz)', 'G4 (392.0 Hz)', 'Chord (all three added)']):
        ax.plot(t, sig, lw=0.8)
        ax.set_ylabel(label, fontsize=8); ax.set_ylim(-1.4, 1.4); ax.grid(alpha=0.3)
    axes[-1].set_xlabel('Time (s)')
    fig.suptitle('Three sine waves → added into a chord')
    plt.tight_layout(); plt.show()
    print('The DFT\'s job is to take the "chord" curve and break it back into the three curves C4/E4/G4.')
except (NotImplementedError, TypeError):
    print('⚠️ Finish the sinusoid() exercise above before running this cell (the chord demo needs your implementation).')


### Fixing it: what a chord looks like at a high enough sample rate

We've seen the illusion in the sr=200 plot above. Now, following the rule of thumb "sample rate > 2 × highest frequency," we raise sr to **8000 Hz** (8000 > 2 × 392, with room to spare). Two small changes are worth noticing:

- **Only the first 20 milliseconds are drawn**. 261.6 Hz has 261 cycles in a second; draw the full second and all the waves squash into one solid band where you can't see anything. Cut it to 20 ms and each wave has only 5–8 cycles, so the shapes are clear at a glance.
- **Each small plot is labeled in its top right with "how many cycles it should have"** (frequency × 0.02 s). You can count the peaks right on the plot to check — and this time, when you check with the formula from whiteboard Question 3, the course's plot passes.

In [ ]:
# Fixing it: raise the sample rate to 8000 Hz and see the real C4/E4/G4
import numpy as np
import matplotlib.pyplot as plt

sr_hi = 8000                  # samples per cycle: 8000/261.6 ≈ 30.6 — this time the formula check passes
dur_hi = 0.02                 # draw only the first 20 ms: 261.6 Hz has 261 cycles per second, a full second would squash into a blob
t_hi = np.arange(round(dur_hi * sr_hi)) / sr_hi

try:
    c4 = sinusoid(t_hi, A=1.0, f=261.6)
    e4 = sinusoid(t_hi, A=0.8, f=329.6)
    g4 = sinusoid(t_hi, A=0.6, f=392.0)
    chord = (c4 + e4 + g4) / 3.0

    fig, axes = plt.subplots(4, 1, figsize=(9, 7), sharex=True)
    for ax, sig, label, f_true in zip(axes,
            [c4, e4, g4, chord],
            ['C4 (261.6 Hz)', 'E4 (329.6 Hz)', 'G4 (392.0 Hz)', 'Chord (all three added)'],
            [261.6, 329.6, 392.0, None]):
        ax.plot(t_hi * 1000, sig, lw=1.0)
        ax.set_ylabel(label, fontsize=8); ax.set_ylim(-1.4, 1.4); ax.grid(alpha=0.3)
        if f_true:
            # in 20 ms there should be f×0.02 cycles — count them right on the plot to check
            ax.set_title(f'In 20 ms, expect {f_true * 0.02:.1f} cycles — count them', fontsize=8, loc='right')
    axes[-1].set_xlabel('Time (ms)')
    fig.suptitle('sr = 8000 Hz: this is what C4/E4/G4 really look like (first 20 ms only)')
    plt.tight_layout(); plt.show()
    print(f'Check passed: samples per cycle = 8000/261.6 ≈ {8000/261.6:.1f} (far more than 2, the waveform can be trusted).')
    print('Only at this sample rate can the DFT really break the chord back into the three bright lines C4/E4/G4.')
except (NotImplementedError, TypeError):
    print('⚠️ Finish the sinusoid() exercise above before running this cell.')

In [ ]:
# Quick check: the same angle, seen as cos, sin, and a complex number at once
# (Borrowing the L05 prop again: np.exp(1j*θ) is that black-box machine that "takes an angle and hands back cos and sin at the same time",
#   z.real = cos(θ), z.imag = sin(θ). How it works is revealed with Euler's formula in L06; here, just keep using it as a black box.)
import numpy as np

for theta in [0, np.pi/2, np.pi, 3*np.pi/2, 2*np.pi]:
    z = np.exp(1j * theta)
    print(f'theta={theta:5.2f} -> cos={z.real:+.1f}, sin={z.imag:+.1f}, complex={z.real:+.1f}{z.imag:+.1f}j')

## Wrap-up

You can now use `sinusoid(t, A, f, phi)` to generate a sine wave with any amplitude, frequency, and phase,
and use `assert np.allclose` to verify the numbers are right.

**Chord = sine waves added together**: add two `sinusoid`s of different frequencies, and two bright lines appear on the spectrogram —
which is exactly the intuition behind the DFT's inverse problem: "given the summed waveform, recover the position and height of each line."

**Next lesson L05**: the magnitude and phase of complex numbers — issue each frequency an ID card and read off its magnitude and phase; this is the FFT's native language.
Every output frequency point (frequency bin) of the FFT is a complex number: **magnitude = how loud that frequency is, phase = timing alignment information**.
Today's A and φ correspond to that complex number's magnitude and phase — **you'll work out the exact correspondence hands-on in the next lesson, L05**; for today, just build the picture of "two quantities, one ID card."

---
⬇️ **Completion check**: you've read the wrap-up; please finish the whiteboard challenge below before ticking the self-review.


## ✏️ Whiteboard challenge: the three ingredients by hand (target: 8 minutes)

Cover the screen and answer on paper:

**Parameters**: A = 2, f = 440 Hz, φ = π/4, sr = 8000 Hz

**Question 1**: Write out the full formula for `x(t)` (plug in the numbers, don't simplify).

**Question 2**: At `t = 1/880` s, what is the exact value of x?
Hint: `sin(2π·440·(1/880) + π/4)` = `sin(π + π/4)` = ?

> 🛟 **Life preserver: forgot the reduction formula?** If that high-school identity **sin(π + x) = −sin(x)** has long since slipped your mind, rebuild it on the spot with the unit circle from Section 1: adding π to the angle means the point **turns half a circle** and ends up directly opposite across the center — both "shadows" (cos and sin) flip to the opposite side at once, so both **change sign**. So sin(π + π/4) = −sin(π/4) = −√2/2 ≈ −0.707. Forgetting the formula is fine; the picture "half a turn → the shadows flip sign" is more reliable than the formula.

**Question 3**: At a frequency of 440 Hz and a sample rate of 8000 Hz, how many samples are there in one cycle? (It doesn't have to be a whole number.)

**Question 4**: If you change φ to 0, what does x(0) become? (And what was x(0) originally, with φ=π/4?)

Once you've worked it out, run the cell below to check your answers.

In [ ]:
# ✏️ Answer-check cell
import numpy as np

A, f, phi, sr = 2.0, 440.0, np.pi/4, 8000

# Q1: formula check (verify your formula with code)
t_test = np.array([0.0, 1/880, 1/440])
x_formula = A * np.sin(2 * np.pi * f * t_test + phi)
print(f"Q1 ✅  x(t) = {A} · sin(2π·{f:.0f}·t + π/4)")
print(f"       x(0)      = {x_formula[0]:.4f}")
print(f"       x(1/880)  = {x_formula[1]:.4f}")

# Q2: check the hand calculation
t2 = 1/880
theta = 2 * np.pi * f * t2 + phi
# 2π·440·(1/880) = 2π·0.5 = π, so θ = π + π/4 = 5π/4
# Why is sin(5π/4) = -√2/2 in the middle step? Two small steps:
#   ① Reduction formula: sin(π + x) = -sin(x) (half a turn, the shadow moves to the opposite side of the center and flips sign)
#      so sin(5π/4) = sin(π + π/4) = -sin(π/4)
#   ② Special angle: sin(π/4) = sin(45°) = √2/2 ≈ 0.707
#   Together: sin(5π/4) = -√2/2, times the amplitude 2 → -√2
x2_theory = 2 * np.sin(5 * np.pi / 4)  # = 2·(-√2/2) = -√2
x2_code   = A * np.sin(2 * np.pi * f * t2 + phi)
assert np.isclose(x2_code, x2_theory, atol=1e-10), f"Q2 mismatch: {x2_code:.6f} vs {x2_theory:.6f}"
print(f"\nQ2 ✅  x(1/880) = 2·sin(5π/4) = 2·(-√2/2) = -√2 ≈ {x2_theory:.4f}  code value {x2_code:.4f}")

# Q3: samples per cycle
samples_per_cycle = sr / f
print(f"\nQ3 ✅  samples per cycle = {sr}/{f:.0f} = {samples_per_cycle:.4f} samples/cycle")

# Q4: x(0) when the phase is 0
x_phi0 = A * np.sin(2 * np.pi * f * 0 + 0)    # = 0
x_phiQ = A * np.sin(2 * np.pi * f * 0 + phi)  # = 2·sin(π/4) = √2
assert np.isclose(x_phi0, 0.0, atol=1e-12)
print(f"\nQ4 ✅  with φ=0, x(0) = {x_phi0:.4f}  with φ=π/4, x(0) = {x_phiQ:.4f} (= √2 ≈ {np.sqrt(2):.4f})")
print("\n🎉 Whiteboard challenge passed! You've internalized working out A / f / φ by hand.")

In [ ]:
# ✏️ Self-review for this lesson
l04_review = {
    "sinusoid_implemented":   None,  # Did you implement sinusoid and pass the assertions? True/False
    "three_params_understood": None, # Can you explain how A/f/φ each affect the waveform? True/False
    "whiteboard_passed":       None, # Did you finish the whiteboard challenge derivation on paper? True/False
    "chord_intuition":         None, # Do you understand "chord = several sine waves added together"? True/False
}

unfilled = [k for k, v in l04_review.items() if v is None]
assert not unfilled, f'Not filled in yet: {unfilled}'
weak = [k for k, v in l04_review.items() if v is False]
if weak:
    print(f'⚠️  Needs more work: {weak}')
else:
    print('✅ L04 complete! On to L05: the geometry of complex numbers')

---

→ **Next lesson** · [L05 · The magnitude and phase of a complex number](L05_complex_numbers.ipynb)

> Next lesson covers **the magnitude and phase of complex numbers**: issue each frequency an ID card — read off magnitude and phase from a complex number; this is the FFT's native language.